<a href="https://colab.research.google.com/github/sunilbarigala25/tcgenerator-colab-gradio/blob/main/tc.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# 🚀 Premium Gradio UAT Test Case Generator - Final Edition
# ============================================================
# Paste this entire code block into a Google Colab cell and run it.

!pip install --quiet gradio openai pandas openpyxl

import gradio as gr
from openai import OpenAI
import json, re, os
import pandas as pd
from datetime import datetime

# -------------------------
# Master prompt (fixed) - User can override
# -------------------------
DEFAULT_MASTER_PROMPT = r"""
Act as a senior QA test designer and generate detailed manual UAT test cases in my organization's standard format.

⚠️ CRITICAL: You MUST output ONLY valid JSON. The system will convert your JSON to Excel (.xlsx) automatically.
DO NOT attempt to create Excel files, markdown tables, or add any explanatory text.

Follow the rules and structure exactly as described below:

---

### FILE FORMAT RULES

The final Excel file will contain these exact column headers:
unique_id | type | step_type | name | teams_udf | testcasetype_udf | testphase_udf | dmnum_udf | testjourney_udf | description | criticality_udf | prerequisites_udf | testdata_udf | circle_udf | step_description

Your JSON must include ALL these fields in EVERY object.

* All columns must exist, even if some cells are empty (use empty string "").
* Each test case begins with a row where type = "test_manual".
* Each step within that test case should be represented by rows where type = "step" and step_type = "simple".
* The last step must contain the expected result written inside step_description.
* Test Case IDs (name) should start from TC001 and increment sequentially.
* testcasetype_udf must always be "UAT" for test_manual rows, "" for step rows.
* testphase_udf must always be "UAT" for test_manual rows, "" for step rows.
* dmnum_udf should contain the provided JIRA ID for test_manual rows, "" for step rows.
* criticality_udf should be assigned as "High", "Medium", or "Low" based on business impact for test_manual rows, "" for step rows.
* step_type must always be "simple" for step rows, "" for test_manual rows.
* Leave teams_udf, testjourney_udf, and circle_udf as "" (empty strings) in all rows.

---

### JSON STRUCTURE EXAMPLE

You MUST output a JSON array following this EXACT pattern:

```json
[
  {
    "unique_id": 1,
    "type": "test_manual",
    "step_type": "",
    "name": "TC001",
    "teams_udf": "",
    "testcasetype_udf": "UAT",
    "testphase_udf": "UAT",
    "dmnum_udf": "JIRA-123",
    "testjourney_udf": "",
    "description": "Verify user can login with valid credentials",
    "criticality_udf": "High",
    "prerequisites_udf": "User account exists in system",
    "testdata_udf": "Username: testuser@example.com, Password: Test@123",
    "circle_udf": "",
    "step_description": ""
  },
  {
    "unique_id": 2,
    "type": "step",
    "step_type": "simple",
    "name": "",
    "teams_udf": "",
    "testcasetype_udf": "",
    "testphase_udf": "",
    "dmnum_udf": "",
    "testjourney_udf": "",
    "description": "",
    "criticality_udf": "",
    "prerequisites_udf": "",
    "testdata_udf": "",
    "circle_udf": "",
    "step_description": "Navigate to the login page"
  },
  {
    "unique_id": 3,
    "type": "step",
    "step_type": "simple",
    "name": "",
    "teams_udf": "",
    "testcasetype_udf": "",
    "testphase_udf": "",
    "dmnum_udf": "",
    "testjourney_udf": "",
    "description": "",
    "criticality_udf": "",
    "prerequisites_udf": "",
    "testdata_udf": "",
    "circle_udf": "",
    "step_description": "Enter valid username in the username field"
  },
  {
    "unique_id": 4,
    "type": "step",
    "step_type": "simple",
    "name": "",
    "teams_udf": "",
    "testcasetype_udf": "",
    "testphase_udf": "",
    "dmnum_udf": "",
    "testjourney_udf": "",
    "description": "",
    "criticality_udf": "",
    "prerequisites_udf": "",
    "testdata_udf": "",
    "circle_udf": "",
    "step_description": "Enter valid password in the password field"
  },
  {
    "unique_id": 5,
    "type": "step",
    "step_type": "simple",
    "name": "",
    "teams_udf": "",
    "testcasetype_udf": "",
    "testphase_udf": "",
    "dmnum_udf": "",
    "testjourney_udf": "",
    "description": "",
    "criticality_udf": "",
    "prerequisites_udf": "",
    "testdata_udf": "",
    "circle_udf": "",
    "step_description": "Click on the Login button"
  },
  {
    "unique_id": 6,
    "type": "step",
    "step_type": "simple",
    "name": "",
    "teams_udf": "",
    "testcasetype_udf": "",
    "testphase_udf": "",
    "dmnum_udf": "",
    "testjourney_udf": "",
    "description": "",
    "criticality_udf": "",
    "prerequisites_udf": "",
    "testdata_udf": "",
    "circle_udf": "",
    "step_description": "Expected Result: User is successfully logged in and redirected to the dashboard page"
  }
]
```

---

### CONTENT RULES

* Each test case must have a clear, concise description in the description field.
* Write detailed, numbered test steps in step_description - be specific about actions.
* The expected result should be included as the last step with "Expected Result:" prefix.
* Include preconditions and test data in their respective columns (prerequisites_udf and testdata_udf).
* Ensure coverage for positive, negative, and edge case scenarios.
* If multiple user roles (e.g., Vi Category Lead, Vi Finance Team, Vi Super Admin) are mentioned in the story, create separate test cases or logically distribute them.
* Avoid combining all roles in one test case unless required.
* Generate comprehensive coverage - AT LEAST 10-15 test cases for non-trivial stories.
* For stories with multiple personas/roles, create dedicated test cases for each role.

---

### OUTPUT INSTRUCTIONS

⚠️ CRITICAL OUTPUT REQUIREMENTS:
* Output ONLY the JSON array - no explanations, no markdown formatting, no text before or after.
* Do NOT wrap in code blocks (```json``` or ```).
* Do NOT say "here is the output" or add any commentary.
* Start directly with [ and end with ].
* Ensure all JSON is properly formatted and parseable.
* All string values must be properly escaped.
* The filename will be: <<JIRA-ID>>_UAT_TCs.xlsx (system generates this automatically).

REMEMBER: You generate JSON. The system converts it to Excel. Do not mention Excel in your output.
"""

# -------------------------
# Helper utilities
# -------------------------
EXPECTED_COLS = [
    "unique_id","type","step_type","name","teams_udf","testcasetype_udf",
    "testphase_udf","dmnum_udf","testjourney_udf","description","criticality_udf",
    "prerequisites_udf","testdata_udf","circle_udf","step_description"
]

def extract_json_from_text(text):
    """Enhanced JSON extraction with multiple strategies"""
    text = text.strip()

    # Strategy 1: Direct JSON parse
    try:
        return json.loads(text)
    except Exception:
        pass

    # Strategy 2: Code block ```json ... ```
    m = re.search(r"```json\s*(.*?)\s*```", text, re.S | re.I)
    if m:
        try:
            return json.loads(m.group(1).strip())
        except Exception:
            pass

    # Strategy 3: Any code block
    m2 = re.search(r"```\s*(.*?)\s*```", text, re.S)
    if m2:
        try:
            return json.loads(m2.group(1).strip())
        except Exception:
            pass

    # Strategy 4: Find JSON-like block (array or object)
    first = min([i for i in (text.find('['), text.find('{')) if i!=-1], default=-1)
    if first != -1:
        for lastchar in (']', '}'):
            last = text.rfind(lastchar)
            if last > first:
                candidate = text[first:last+1]
                try:
                    return json.loads(candidate)
                except Exception:
                    continue

    return None

def create_fallback_json_from_story(story, jira_id):
    """
    When API quota is exhausted, generate a structured JSON fallback
    based on the user story content (no AI involved)
    """
    lines = [ln.strip() for ln in story.splitlines() if ln.strip()]

    # Try to extract title/description
    title = lines[0] if lines else "Test Case from User Story"
    if len(title) > 200:
        title = title[:200]

    # Extract acceptance criteria if present
    acceptance_criteria = []
    in_criteria = False
    for line in lines:
        if "acceptance criteria" in line.lower() or "ac:" in line.lower():
            in_criteria = True
            continue
        if in_criteria and line.startswith("-"):
            acceptance_criteria.append(line.lstrip("- ").strip())

    # Generate basic test cases
    fallback_json = []
    tc_count = max(3, len(acceptance_criteria)) if acceptance_criteria else 5

    for i in range(1, tc_count + 1):
        unique_id_start = (i - 1) * 5 + 1

        # Test case header
        if acceptance_criteria and i <= len(acceptance_criteria):
            description = f"Verify: {acceptance_criteria[i-1]}"
        else:
            description = f"{title} - Scenario {i}"

        fallback_json.append({
            "unique_id": unique_id_start,
            "type": "test_manual",
            "step_type": "",
            "name": f"TC{str(i).zfill(3)}",
            "teams_udf": "",
            "testcasetype_udf": "UAT",
            "testphase_udf": "UAT",
            "dmnum_udf": jira_id or "",
            "testjourney_udf": "",
            "description": description,
            "criticality_udf": "Medium",
            "prerequisites_udf": "System access and test environment available",
            "testdata_udf": "Valid test data as per test scenario",
            "circle_udf": "",
            "step_description": ""
        })

        # Test steps
        steps = [
            "Review the user story requirements and acceptance criteria",
            "Prepare necessary test data and test environment",
            "Execute the test scenario as per the user story",
            "Verify the actual results against expected behavior",
            "Expected Result: System behaves as described in the user story and acceptance criteria are met"
        ]

        for step_idx, step_text in enumerate(steps, start=1):
            fallback_json.append({
                "unique_id": unique_id_start + step_idx,
                "type": "step",
                "step_type": "simple",
                "name": "",
                "teams_udf": "",
                "testcasetype_udf": "",
                "testphase_udf": "",
                "dmnum_udf": "",
                "testjourney_udf": "",
                "description": "",
                "criticality_udf": "",
                "prerequisites_udf": "",
                "testdata_udf": "",
                "circle_udf": "",
                "step_description": step_text
            })

    return fallback_json

def json_to_excel(json_data, jira_id):
    """Convert JSON to Excel and return file path"""
    # Ensure all rows have all required columns
    rows = []
    for item in json_data:
        row = {}
        for col in EXPECTED_COLS:
            row[col] = item.get(col, "")
        rows.append(row)

    df = pd.DataFrame(rows, columns=EXPECTED_COLS)

    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    filename = f"{(jira_id or 'NOJIRA')}_UAT_TCs_{timestamp}.xlsx"
    filepath = os.path.join("/content", filename)

    df.to_excel(filepath, index=False)
    return filepath

# -------------------------
# Main generation function
# -------------------------
def generate_tcs(api_key, model_choice, custom_prompt_toggle, custom_prompt, jira_id, story_text, txt_file, progress=gr.Progress()):
    """
    Enhanced test case generation with quota fallback logic
    """
    progress(0, desc="🔍 Validating inputs...")

    # Validation 1: API Key
    if not api_key or len(api_key.strip()) == 0:
        return ("❌ Please enter an OpenAI API key.", None, None)

    api_key = api_key.strip()
    if not api_key.startswith('sk-'):
        return ("⚠️ API key should start with 'sk-'. Please check your key.", None, None)

    progress(0.1, desc="📄 Processing input story...")

    # Validation 2: Story source
    if txt_file is not None:
        try:
            if isinstance(txt_file, bytes):
                story = txt_file.decode("utf-8")
            else:
                story = txt_file
        except Exception as e:
            return (f"❌ Error reading uploaded file: {e}", None, None)
    else:
        story = story_text or ""

    if not story.strip():
        return ("❌ Please provide a JIRA story (paste text or upload .txt file).", None, None)

    # Validation 3: Story length
    if len(story.strip()) < 50:
        return ("⚠️ Story seems too short. Please provide more details for better test case generation.", None, None)

    progress(0.2, desc="✍️ Composing prompt...")

    # Choose prompt: custom or default
    if custom_prompt_toggle and custom_prompt.strip():
        base_prompt = custom_prompt.strip()
        progress(0.25, desc="📝 Using custom prompt...")
    else:
        base_prompt = DEFAULT_MASTER_PROMPT

    # Compose final prompt
    prompt = base_prompt + "\n\n---\nJIRA ID: " + (jira_id or "") + "\n\nUser Story:\n" + story

    progress(0.3, desc="🤖 Calling OpenAI API...")

    # Initialize OpenAI client
    try:
        client = OpenAI(api_key=api_key)
    except Exception as e:
        return (f"❌ Failed to initialize OpenAI client: {e}", None, None)

    # Try to call API
    try:
        progress(0.4, desc="⏳ Generating test cases...")

        resp = client.chat.completions.create(
            model=model_choice,
            messages=[
                {"role":"system", "content":"You are a senior QA test designer. You MUST output ONLY valid JSON arrays. Never generate Excel files, markdown tables, or explanatory text. Your output will be automatically converted to Excel by the system."},
                {"role":"user", "content": prompt}
            ],
            temperature=0.1,
            max_tokens=16000
        )

        progress(0.7, desc="📊 Parsing AI response...")

        model_text = resp.choices[0].message.content.strip()
        parsed = extract_json_from_text(model_text)

        if parsed is None:
            # AI returned non-JSON - show raw output
            return (
                f"⚠️ AI returned non-JSON output. Raw response shown below.\n\n{model_text[:2000]}...",
                None,
                model_text
            )

        progress(0.8, desc="📋 Converting to Excel...")

        # Successfully parsed JSON - convert to Excel
        if isinstance(parsed, dict):
            parsed = [parsed]

        excel_path = json_to_excel(parsed, jira_id)

        progress(1.0, desc="✅ Complete!")

        json_preview = json.dumps(parsed, indent=2)
        if len(json_preview) > 20000:
            json_preview = json_preview[:20000] + "\n\n... (truncated for display)"

        tc_count = len([x for x in parsed if x.get("type") == "test_manual"])

        return (
            f"✅ Successfully generated {tc_count} test cases with {len(parsed)} total rows!\n\n📥 Excel file is ready for download below.",
            excel_path,
            json_preview
        )

    except Exception as e:
        error_msg = str(e).lower()

        progress(0.5, desc="⚠️ API error detected...")

        # Check if quota/billing issue
        if "quota" in error_msg or "insufficient" in error_msg or "billing" in error_msg or "rate_limit" in error_msg:
            progress(0.6, desc="🔄 Quota exhausted - generating fallback JSON...")

            # QUOTA EXHAUSTED: Generate JSON manually from story
            fallback_json = create_fallback_json_from_story(story, jira_id)

            progress(0.8, desc="📋 Converting fallback JSON to Excel...")

            # Convert to Excel
            excel_path = json_to_excel(fallback_json, jira_id)

            progress(1.0, desc="✅ Fallback complete!")

            json_preview = json.dumps(fallback_json, indent=2)
            if len(json_preview) > 20000:
                json_preview = json_preview[:20000] + "\n\n... (truncated for display)"

            tc_count = len([x for x in fallback_json if x.get("type") == "test_manual"])

            return (
                f"🚫 API QUOTA EXHAUSTED\n\n⚠️ Original Error: {str(e)[:500]}\n\n✅ FALLBACK MODE ACTIVATED:\n- Generated {tc_count} template test cases from your story\n- JSON created without AI (quota-free)\n- Converted to Excel format\n\n📥 Excel file ready for download.\n\n💡 To get AI-generated test cases:\n1. Use a different API key, or\n2. Wait for quota reset, or\n3. Add billing to your OpenAI account",
                excel_path,
                json_preview
            )
        else:
            # Other API errors
            progress(0.6, desc="⚠️ API error - generating fallback...")

            fallback_json = create_fallback_json_from_story(story, jira_id)
            excel_path = json_to_excel(fallback_json, jira_id)

            progress(1.0, desc="⚠️ Fallback complete")

            json_preview = json.dumps(fallback_json, indent=2)

            return (
                f"⚠️ API ERROR\n\nError: {str(e)[:500]}\n\n✅ Created fallback test cases from your story.\n\n📥 Excel file ready for download.",
                excel_path,
                json_preview
            )

# -------------------------
# Material-3 CSS
# -------------------------
M3_CSS = """
body {
    background: linear-gradient(135deg, #f5f7fa 0%, #c3cfe2 100%);
    font-family: 'Google Sans', 'Segoe UI', Roboto, Arial, sans-serif;
    color: #1f2937;
}
.gradio-container {
    padding: 24px !important;
    max-width: 1400px;
    margin: 0 auto;
}
.gr-button {
    border-radius: 12px !important;
    font-weight: 600 !important;
    transition: all 0.2s ease !important;
}
.gr-button:hover {
    transform: translateY(-2px);
    box-shadow: 0 6px 16px rgba(0,0,0,0.12);
}
#generate-btn {
    background: linear-gradient(135deg, #667eea 0%, #764ba2 100%) !important;
    color: white !important;
    border: none !important;
}
.header {
    font-size: 28px;
    font-weight: 700;
    margin-bottom: 8px;
    background: linear-gradient(135deg, #667eea 0%, #764ba2 100%);
    -webkit-background-clip: text;
    -webkit-text-fill-color: transparent;
}
"""

# -------------------------
# Build Gradio UI
# -------------------------
with gr.Blocks(css=M3_CSS, title="🚀 UAT Test Case Generator", theme=gr.themes.Soft()) as demo:

    gr.Markdown("""
    # 🚀 UAT Test Case Generator
    ### Enterprise-grade test case generation powered by OpenAI
    *AI generates JSON → System converts to Excel → You download*
    """)

    with gr.Row():
        with gr.Column(scale=3):
            # API Configuration
            gr.Markdown("### 🔑 API Configuration")
            with gr.Row():
                api_key = gr.Textbox(
                    value="",
                    placeholder="sk-proj-...",
                    type="password",
                    label="OpenAI API Key",
                    scale=3
                )
                model_choice = gr.Dropdown(
                    choices=["gpt-4o-mini", "gpt-4o", "gpt-4-turbo"],
                    value="gpt-4o-mini",
                    label="Model",
                    scale=1
                )

            # Custom Prompt Option
            gr.Markdown("### 🎯 Prompt Configuration")
            custom_prompt_toggle = gr.Checkbox(
                label="Use Custom Prompt (advanced users)",
                value=False,
                info="Enable this to write your own system prompt"
            )
            custom_prompt = gr.Textbox(
                label="Custom System Prompt",
                placeholder="Write your custom prompt here... (Leave empty to use default optimized prompt)",
                lines=8,
                visible=False,
                info="Your prompt should instruct the AI to output JSON in the required format"
            )

            def toggle_custom_prompt(checked):
                return gr.update(visible=checked)

            custom_prompt_toggle.change(
                fn=toggle_custom_prompt,
                inputs=[custom_prompt_toggle],
                outputs=[custom_prompt]
            )

            gr.Markdown("### 📝 Input Your User Story")

            with gr.Tab("📋 Paste Story"):
                jira_field = gr.Textbox(
                    label="JIRA ID (optional)",
                    placeholder="e.g., ECOM-1234, LRECOM-9770",
                    lines=1
                )
                story_field = gr.Textbox(
                    label="User Story",
                    placeholder="As a [role], I want to [action] so that [benefit]...\n\nAcceptance Criteria:\n- ...",
                    lines=12
                )

            with gr.Tab("📤 Upload File"):
                upload_file = gr.File(
                    label="Upload .txt file with user story",
                    file_count="single",
                    type="binary",
                    file_types=[".txt"]
                )
                gr.Markdown("*📌 Uploaded file will override pasted text*")

            with gr.Row():
                generate_btn = gr.Button(
                    "🚀 Generate Test Cases",
                    variant="primary",
                    size="lg",
                    elem_id="generate-btn"
                )
                clear_btn = gr.Button(
                    "🔁 Clear All",
                    variant="secondary",
                    size="lg"
                )

        with gr.Column(scale=2):
            status_output = gr.Textbox(
                label="📊 Status & Messages",
                interactive=False,
                lines=12
            )

            download_link = gr.File(
                label="📥 Download Generated Excel",
                interactive=False
            )

            with gr.Accordion("🔍 Raw JSON Preview", open=False):
                raw_json_out = gr.Code(
                    label="",
                    language="json",
                    lines=15,
                    interactive=False
                )

    gr.Markdown("""
    ---
    <div style='text-align: center; color: #6b7280; font-size: 12px;'>
    🔒 Your API key is never stored | 🤖 Powered by OpenAI | 🎨 Built with Gradio<br>
    💡 <b>Quota Exhausted?</b> No problem - fallback mode generates JSON from your story and converts to Excel automatically!
    </div>
    """)

    # Event handlers
    generate_btn.click(
        fn=generate_tcs,
        inputs=[api_key, model_choice, custom_prompt_toggle, custom_prompt, jira_field, story_field, upload_file],
        outputs=[status_output, download_link, raw_json_out]
    )

    def clear_all():
        return "", "gpt-4o-mini", False, "", "", "", None, "", None, ""

    clear_btn.click(
        fn=clear_all,
        inputs=[],
        outputs=[api_key, model_choice, custom_prompt_toggle, custom_prompt, jira_field, story_field, upload_file, status_output, download_link, raw_json_out]
    )

print("\n" + "="*60)
print("🚀 UAT TEST CASE GENERATOR - FINAL EDITION")
print("="*60)
print("✅ Files download directly to your system")
print("✅ No TC count limit - generates comprehensive coverage")
print("✅ Quota fallback: Auto-generates JSON → Excel")
print("🌐 Launching Gradio interface...")
print("="*60 + "\n")

demo.launch(share=True, show_error=True)